<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ami-null/python-course/blob/main/07_SUPPLEMENTARY_function-arguments.ipynb)

# Function Arguments - Advanced Topics

## Learning Objectives

By the end of this notebook, you should be able to:

- Write functions that accept any number of positional arguments using `*args`
- Write functions that accept any number of keyword arguments using `**kwargs`
- Unpack lists and dictionaries at function call sites using `*` and `**`
- Explain why mutable default arguments are dangerous and write the correct alternative using `None` as a sentinel
- Define keyword-only parameters using a bare `*` in the function signature
- Recall the full order in which Python requires parameters to appear in a signature

## Context

This notebook supplements `07_functions.ipynb`. It assumes you are comfortable defining functions, using default arguments, and understanding how Python passes values to parameters. The gradebook theme from that notebook continues here.

---
## 1. `*args` - Variable Positional Arguments

### What `*args` Does

When Python sees a `*` before a parameter name in a function signature, it does not treat that parameter as a single value. Instead, it collects all remaining positional arguments - however many the caller provides - and packs them into a **tuple**. The function then receives one tuple object containing all of those values.

The name `args` is a widely followed convention, but it is not enforced by Python. The `*` is what triggers the collection behaviour. You could write `*scores`, `*values`, or `*items` and the result would be identical.

Because the collected arguments arrive as a plain tuple, everything you can do with a tuple works on `args` inside the function: iterate over it, pass it to `len()`, index into it, or hand it off to `sum()` or another function.

A function that uses `*args` can also be called with zero extra arguments, in which case the tuple is simply empty.

In [ ]:
def show_args(*args):
    # print the type first so it is clear that args is a tuple, not a list
    print(type(args))
    print(args)

show_args(10, 20, 30)
show_args("alice", "bob")
show_args()    # called with no arguments - args is an empty tuple

<class 'tuple'>
(10, 20, 30)
<class 'tuple'>
('alice', 'bob')
<class 'tuple'>
()


In [ ]:
def compute_total(*scores):
    # sum() works on any iterable, including the tuple that *args produces
    return sum(scores)

print(compute_total(80, 90, 75))
print(compute_total(100))    # works with a single argument
print(compute_total())       # sum of an empty sequence is 0

245
100
0


### Combining `*args` with Regular Parameters

Regular parameters can appear before `*args` in the signature. Python fills them first, in order, and then packs everything that remains into the `*args` tuple. This means that if a function has two regular parameters before `*args`, the first two positional arguments are always consumed by those parameters, and only the third argument onward goes into the tuple.

Parameters that appear *after* `*args` cannot be filled positionally at all - they must be supplied as keyword arguments. This is an important rule that is covered in detail in Section 5.

In [ ]:
def log_event(event_name, *details):
    # event_name always takes the first argument, no matter how many are given
    # details collects everything after it into a tuple
    print(f"Event: {event_name}")
    for i, detail in enumerate(details, 1):
        print(f"  {i}. {detail}")

log_event("Login", "user=alice", "ip=192.168.1.1", "timestamp=2024-01-15")
print()
log_event("Logout")    # details will be an empty tuple

Event: Login
  1. user=alice
  2. ip=192.168.1.1
  3. timestamp=2024-01-15

Event: Logout


#### Gradebook: Averaging an Arbitrary Number of Component Scores

In some courses a final grade comes from two exams; in others it comes from four assignments and a project. A function using `*args` handles any number of components without requiring the signature to change.

In [ ]:
def compute_final(*component_scores):
    # guard against an empty call so we do not divide by zero
    if len(component_scores) == 0:
        return 0.0
    return sum(component_scores) / len(component_scores)

print(compute_final(85, 90))              # two components
print(compute_final(78, 82, 91, 88))      # four components
print(compute_final(100))                 # single component

87.5
84.75
100.0


---
### Exercises - Section 1

**Exercise 1**

Write a function `product(*values)` that returns the product of all given values. `product(2, 3, 4)` should return `24`. Decide what the function should return when called with no arguments and explain your reasoning in a comment.

In [ ]:
# your code here

**Exercise 2 (builds on `compute_final` from this section)**

Write a function `class_average(*student_averages)` that takes any number of individual student averages and returns the overall class average. Use the same guard against an empty call that `compute_final` uses.

In [ ]:
# your code here

---
## 2. `**kwargs` - Variable Keyword Arguments

### What `**kwargs` Does

When Python sees `**` before a parameter name in a function signature, it collects all keyword arguments that do not match any explicitly named parameter and packs them into a **dictionary**. The keys of this dictionary are the argument names as strings; the values are the corresponding values passed by the caller.

The name `kwargs` is conventional. The `**` is what triggers collection. You could write `**options`, `**config`, or `**fields` - the behaviour would be identical.

`**kwargs` is useful in two main situations. The first is building a flexible interface where the full set of valid keyword arguments is not known at definition time - for example, a function that constructs database records with varying fields. The second is forwarding keyword arguments from one function to another without listing them all explicitly, which comes up often when wrapping library functions.

Like `*args`, `**kwargs` can receive zero keyword arguments, in which case the dictionary is empty.

In [ ]:
def show_kwargs(**kwargs):
    # print the type first to confirm kwargs is a dict
    print(type(kwargs))
    print(kwargs)

show_kwargs(name="Alice", score=85, year="2nd")
print()
show_kwargs()    # called with no keyword arguments - kwargs is an empty dict

<class 'dict'>
{'name': 'Alice', 'score': 85, 'year': '2nd'}

<class 'dict'>
{}


In [ ]:
def display_info(**fields):
    # iterating over .items() gives both the key (field name) and the value
    for key, value in fields.items():
        print(f"{key}: {value}")

display_info(name="Alice", subject="Mathematics", grade="A")

name: Alice
subject: Mathematics
grade: A


### Combining Regular Parameters, `*args`, and `**kwargs`

All three can appear in one signature. The order must always be: regular parameters, then `*args`, then `**kwargs`. Python fills them left to right: regular parameters take positional arguments in order, `*args` collects any remaining positional arguments, and `**kwargs` collects any remaining keyword arguments.

Any deviation from this order raises a `SyntaxError` at definition time.

In [ ]:
def mixed(required, *args, **kwargs):
    print(f"required : {required}")
    print(f"args     : {args}")
    print(f"kwargs   : {kwargs}")

mixed("hello", 1, 2, 3, name="Alice", score=90)

required : hello
args     : (1, 2, 3)
kwargs   : {'name': 'Alice', 'score': 90}


#### Gradebook: Flexible Student Record Creation

Different parts of a school system may need different fields on a student record. Using `**kwargs` lets a single function handle any combination of fields without needing a long, fixed list of parameters.

In [ ]:
def create_student_record(name, **attributes):
    # name is always required; every other field is optional and open-ended
    # dict.update() copies all key-value pairs from attributes into record
    record = {"name": name}
    record.update(attributes)
    return record

print(create_student_record("Alice", score=85, year="2nd", subject="Math"))
print(create_student_record("Bob", score=72))

{'name': 'Alice', 'score': 85, 'year': '2nd', 'subject': 'Math'}
{'name': 'Bob', 'score': 72}


---
### Exercises - Section 2

**Exercise 3**

Write a function `build_query(**filters)` that takes any number of keyword arguments and returns a single string in the format `"key1=value1 AND key2=value2"`. For example, `build_query(name="Alice", grade="A")` should return `"name=Alice AND grade=A"`. The order of the pairs should match the order they were passed.

In [ ]:
# your code here

**Exercise 4 (builds on Exercise 3)**

Write a function `filter_students(students, **criteria)` that takes a list of student dicts and any number of keyword filters, and returns only the students whose fields match **all** of the given criteria. For example:

```python
students = [
    {"name": "Alice", "grade": "A", "year": "2nd"},
    {"name": "Bob",   "grade": "B", "year": "2nd"},
    {"name": "Carol", "grade": "A", "year": "3rd"},
]
filter_students(students, grade="A", year="2nd")
# should return only Alice
```

In [ ]:
# your code here

---
## 3. Argument Unpacking at Call Sites

### The Inverse of `*args` and `**kwargs`

The `*` and `**` operators work in both directions. In a function **definition**, they collect multiple arguments into a single container - `*args` packs positional arguments into a tuple, `**kwargs` packs keyword arguments into a dict. At a function **call site**, the same operators work in reverse: `*` unpacks an iterable into individual positional arguments, and `**` unpacks a dictionary into keyword arguments.

Understanding both sides - packing in definitions, unpacking at call sites - gives a complete picture of how Python moves data in and out of functions.

### Unpacking Positional Arguments with `*`

If you have a list, tuple, or any other iterable and you want to pass its elements as individual positional arguments, prefix the variable with `*` at the call site. Python will spread each element out as if you had typed them all out separately.

This solves a specific problem: a function expects three separate values, but you only have a list. Without unpacking, passing the list would give the function a single list object rather than three values.

In [ ]:
def add(a, b, c):
    return a + b + c

In [ ]:
values = [10, 20, 30]

In [ ]:
# UNCOMMENT TO SEE THE ERROR
# add(values)
# TypeError: add() missing 2 required positional arguments
# Python sees one argument (the list), not three

In [ ]:
# with * unpacking, the list is spread into three separate positional arguments
print(add(*values))

60


In [ ]:
# * unpacking works with any iterable: tuples, ranges, generator expressions
coordinates = (3, 7, 2)
print(add(*coordinates))

12


In [ ]:
list(range(1, 4))

[1, 2, 3]

In [ ]:
# range(1, 4) produces the values 1, 2, 3 - unpacking passes them individually
print(add(*range(1, 4)))

6


### Unpacking Keyword Arguments with `**`

If you have a dictionary whose keys match a function's parameter names, you can unpack it at the call site with `**`. Python treats each key-value pair as a keyword argument - exactly as if you had typed `key=value` for each pair manually.

This is particularly useful when you are building up a set of arguments programmatically - constructing a dict based on some logic and then calling a function with it, without needing to know in advance which keys are present.

In [ ]:
def greet(first, last, title=""):
    if title:
        return f"Hello, {title} {last}"
    return f"Hello, {first} {last}"

person = {"first": "Alan", "last": "Turing", "title": "Dr."}
# the three key-value pairs become three keyword arguments
print(greet(**person))

# you can mix explicit arguments with ** unpacking
overrides = {"last": "Lovelace", "title": "Countess"}
print(greet("Ada", **overrides))

Hello, Dr. Turing
Hello, Countess Lovelace


### Combining `*` and `**` at a Single Call Site

Both operators can appear in the same call. Python processes `*` unpacking first (filling positional slots), then `**` unpacking (filling keyword slots). You can also mix unpacking with explicit arguments, as long as no argument is supplied more than once.

In [ ]:
def report(name, *scores, label="Final"):
    avg = sum(scores) / len(scores)
    print(f"{label} - {name}: {avg:.2f}")

student_scores = [85, 90, 78]
options = {"label": "Midterm"}

# * spreads the list into positional score arguments
# ** spreads the dict into the label keyword argument
report("Alice", *student_scores, **options)

Midterm - Alice: 84.33


#### Gradebook: Calling Functions from Stored Records

Student data is often stored as dictionaries - loaded from a file, a database, or an API response. Unpacking with `**` lets you call a function directly from a stored record without manually extracting each field.

In [ ]:
def grade_summary(name, scores):
    avg = sum(scores) / len(scores)
    return f"{name}: {avg:.2f}"

students = [
    {"name": "Alice", "scores": [85, 90, 78]},
    {"name": "Bob",   "scores": [72, 68, 74]},
]

for student in students:
    # ** unpacks the dict so name= and scores= are filled from the record
    print(grade_summary(**student))

Alice: 84.33
Bob: 71.33


---
### Exercises - Section 3

**Exercise 5**

You have the list `values = [4, 9, 16]`. Use `*` unpacking to pass these three values to the built-in `print()` function with the keyword argument `sep=" | "`. The output should be `4 | 9 | 16`.

In [ ]:
values = [4, 9, 16]
# your code here

**Exercise 6 (builds on `create_student_record()` from Section 2)**

You have a list of student dicts:

```python
students = [
    {"name": "Alice", "score": 85, "year": "2nd"},
    {"name": "Bob",   "score": 72, "year": "3rd"},
]
```

Use a loop and `**` unpacking to call `create_student_record()` on each dict and print the resulting records.

In [ ]:
students = [
    {"name": "Alice", "score": 85, "year": "2nd"},
    {"name": "Bob",   "score": 72, "year": "3rd"},
]
# your code here

---
## 4. Mutable Default Arguments

### The Problem

Python evaluates default argument values **once**, at the time the `def` statement runs - not each time the function is called. For immutable defaults like integers, strings, or tuples, this makes no practical difference because their values cannot change. Sharing the same immutable object across all calls is harmless.

For mutable defaults like lists or dictionaries, this is a serious trap. The same list or dictionary object is reused across every call that relies on the default. If one call modifies that object - by appending to the list, or adding a key to the dict - the modification persists. The next call that uses the default sees the already-modified object, not a fresh one. This compounds with every call.

This surprises many programmers because the code looks like it should create a fresh list or dict each time. It does not. The default object is created once and lives on the function for the function's entire lifetime.

In [ ]:
def add_score(score, score_list=[]):
    # this list is created ONCE when Python reads the def statement
    # every call that omits score_list shares the exact same list object
    score_list.append(score)
    return score_list

# each call looks independent but they all share the same list
print(add_score(85))    # expected [85]  - actual [85]
print(add_score(90))    # expected [90]  - actual [85, 90]
print(add_score(78))    # expected [78]  - actual [85, 90, 78]

[85]
[85, 90]
[85, 90, 78]


### Why This Happens

Functions in Python are objects. When Python executes a `def` statement, it creates a function object and stores the default values as an attribute of that object. You can inspect this directly using `__defaults__`, which holds a tuple of the default values.

Because the list lives on the function object rather than being created at call time, every call that uses the default is operating on the same object.

In [ ]:
# the default list is stored as part of the function object itself
print(add_score.__defaults__)

# calling the function again modifies the object stored in __defaults__
add_score(100)
print(add_score.__defaults__)    # the list has grown - the default has mutated

([85, 90, 78],)
([85, 90, 78, 100],)


### The Fix: Use `None` as default

---
## 5. Keyword-only Arguments

### Forcing Keyword Use with a Bare `*`

Placing a bare `*` - with no name after it - in the parameter list acts as a dividing line. Every parameter to the right of the `*` must be supplied as a keyword argument when the function is called. It cannot be filled positionally, no matter how many positional arguments the caller provides.

This is distinct from `*args`. A bare `*` does not collect any arguments at all - it has no tuple associated with it. Its sole purpose is to mark the keyword-only boundary. Parameters to its left can be filled either positionally or by keyword (unless they are positional-only, covered in Section 6). Parameters to its right can only ever be supplied by keyword.

The practical reason to enforce this is readability. Consider two calls to the same function:

```python
format_report("Alice", True, False, 2)
format_report("Alice", bold=True, border=False, copies=2)
```

The first call is opaque - `True`, `False`, and `2` mean nothing without reading the function definition. The second call is self-documenting. Making `bold`, `border`, and `copies` keyword-only ensures callers cannot produce the opaque form even by accident.

In [ ]:
def create_report(name, *, include_average, include_rank):
    # include_average and include_rank are keyword-only
    # the bare * means there are no positional slots available for them
    parts = [f"Student: {name}"]
    if include_average:
        parts.append("Average: included")
    if include_rank:
        parts.append("Rank: included")
    return " | ".join(parts)

print(create_report("Alice", include_average=True, include_rank=False))
print(create_report("Bob",   include_average=True, include_rank=True))

Student: Alice | Average: included
Student: Bob | Average: included | Rank: included


In [ ]:
# UNCOMMENT TO SEE THE ERROR
# create_report("Alice", True, False)
# TypeError: create_report() takes 1 positional argument but 3 were given
# Python has no positional slots for True and False because of the bare *

### Keyword-only Parameters with Default Values

Keyword-only parameters can have default values just like regular parameters. When they have defaults, the caller is not required to supply them - but if the caller does supply them, they must use the keyword form. There is no way to bypass this with a positional argument.

In [ ]:
def format_grade(score, *, passing_mark=50, prefix="Grade"):
    status = "pass" if score >= passing_mark else "fail"
    return f"{prefix}: {score} ({status})"

print(format_grade(75))                                       # both defaults used
print(format_grade(75, passing_mark=60))                      # one default overridden
print(format_grade(75, prefix="Score", passing_mark=60))      # both overridden

# UNCOMMENT TO SEE THE ERROR
# format_grade(75, 60)
# TypeError: format_grade() takes 1 positional argument but 2 were given

Grade: 75 (pass)
Grade: 75 (pass)
Score: 75 (pass)


### Combining `*args` with Keyword-only Parameters

When `*args` is present in a signature, it consumes all remaining positional arguments. Any parameter listed after `*args` has no positional slot left, which makes it automatically keyword-only. The bare `*` rule and the `*args` rule are two manifestations of the same underlying principle: the keyword-only boundary starts wherever the positional slots run out.

This is why, in Section 1, the note said that parameters after `*args` cannot be filled positionally. They are implicitly keyword-only.

In [ ]:
def summarise(*scores, label="Summary", passing_mark=50):
    # label and passing_mark appear after *args, so they are keyword-only
    # there are no positional slots for them because *args took them all
    avg = sum(scores) / len(scores) if scores else 0.0
    status = "pass" if avg >= passing_mark else "fail"
    print(f"{label}: avg={avg:.2f} ({status})")

summarise(80, 90, 75, label="Midterm")
summarise(40, 35, label="Quiz", passing_mark=40)
summarise(label="Empty test")    # scores tuple will be empty

Midterm: avg=81.67 (pass)
Quiz: avg=37.50 (fail)
Empty test: avg=0.00 (fail)


#### Gradebook: A Configurable Student Report

In [ ]:
def print_student_report(name, *scores, show_grade=True, show_status=True, passing_mark=50):
    # name takes the first positional argument
    # *scores takes all remaining positional arguments
    # show_grade, show_status, passing_mark are keyword-only (after *scores)
    avg = sum(scores) / len(scores) if scores else 0.0

    line = f"{name} | Average: {avg:.2f}"

    if show_grade:
        if avg >= 90:   grade = "A"
        elif avg >= 80: grade = "B"
        elif avg >= 70: grade = "C"
        elif avg >= 60: grade = "D"
        else:           grade = "F"
        line += f" | Grade: {grade}"

    if show_status:
        line += f" | {'Pass' if avg >= passing_mark else 'Fail'}"

    print(line)

print_student_report("Alice", 85, 90, 78)
print_student_report("Bob",   45, 50, 40, show_grade=False, passing_mark=45)
print_student_report("Carol", 92, 88, 95, show_status=False)

Alice | Average: 84.33 | Grade: B | Pass
Bob | Average: 45.00 | Pass
Carol | Average: 91.67 | Grade: A


---
### Exercises - Section 5

**Exercise 9**

Write a function `divide(a, b, *, safe=True)` where `a` and `b` are regular positional parameters and `safe` is keyword-only with a default of `True`.

- When `safe=True`, return `None` instead of raising a `ZeroDivisionError` if `b` is 0.
- When `safe=False`, let the error propagate normally.

Test all three cases: normal division, division by zero with `safe=True`, and division by zero with `safe=False` (comment out the last one).

In [ ]:
# your code here

**Exercise 10 (builds on Exercise 9 and `letter_grade()`, `is_passing()` from `07_functions.ipynb`)**

Write a function `student_stats(*scores, passing_mark=50, verbose=True)`.

- When `verbose=True`, print the minimum score, maximum score, average, letter grade (using `letter_grade()` from `07_functions.ipynb`), and pass/fail status (using `is_passing()` from `07_functions.ipynb`).
- When `verbose=False`, print only the average and pass/fail status.

All parameters after `*scores` must be keyword-only.

In [ ]:
# your code here

---
## 6. The Full Argument Order

### The Five Categories

Python enforces a strict ordering of parameter types in any function signature. Placing them out of order raises a `SyntaxError` before the code even runs.

The complete order is:

1. **Positional-only** (before `/`) - must be supplied positionally; the caller cannot use the parameter name
2. **Regular** (no marker) - can be supplied either positionally or by keyword
3. **`*args`** or a **bare `*`** - `*args` collects extra positional arguments into a tuple; bare `*` marks the keyword-only boundary without collecting anything
4. **Keyword-only** (after `*` or `*args`) - must be supplied by keyword; no positional slot is available
5. **`**kwargs`** - collects all remaining keyword arguments into a dict

In everyday code, most functions use only one or two of these categories. The full form is rare in user-written code but appears regularly in the standard library and third-party packages. Reading library documentation requires being able to recognise all five.

### Positional-only Parameters (`/`)

A `/` in the signature marks the end of the positional-only section. Parameters to the left of `/` cannot be supplied by keyword - the caller must pass them positionally. This feature was added in Python 3.8 and is used mainly in the standard library and C extensions.

You will see it in built-in function signatures such as `len(obj, /)`, which means `obj` cannot be passed as `len(obj=my_list)`. For most user-defined functions you will not write `/` yourself. The important skill is recognising it when you encounter it in documentation.

In [ ]:
# a function using all five parameter categories at once
def full_example(pos_only, /, regular, *args, kw_only, **kwargs):
    print(f"pos_only  : {pos_only}")    # filled by position, cannot use keyword
    print(f"regular   : {regular}")     # can be positional or keyword
    print(f"args      : {args}")        # tuple of any extra positional arguments
    print(f"kw_only   : {kw_only}")     # must be keyword
    print(f"kwargs    : {kwargs}")      # dict of any extra keyword arguments

full_example(1, 2, 3, 4, kw_only="required", extra="also keyword", another=99)

pos_only  : 1
regular   : 2
args      : (3, 4)
kw_only   : required
kwargs    : {'extra': 'also keyword', 'another': 99}


In [ ]:
# UNCOMMENT TO SEE THE ERROR
# pos_only cannot be passed by keyword - the / enforces positional-only
# full_example(pos_only=1, regular=2, kw_only="x")
# TypeError: full_example() got some positional-only arguments passed as keyword arguments

### Quick Reference

| Category | Syntax in signature | Can be positional? | Can be keyword? |
|---|---|:---:|:---:|
| Positional-only | before `/` | yes | no |
| Regular | no marker | yes | yes |
| Extra positional | `*args` | collected into tuple | - |
| Keyword-only | after `*` or `*args` | no | yes |
| Extra keyword | `**kwargs` | - | collected into dict |

---
## Where to Go Next

The following topics connect directly to what is covered here. Each is large enough to deserve its own notebook.

- **Closures** - functions that capture and remember variables from their enclosing scope, even after that scope has finished executing. Understanding how Python looks up variable names is a prerequisite.
- **Decorators** - functions that wrap other functions to add or modify behaviour. They rely on the fact that functions are objects and can be passed as arguments and returned from other functions.
- **`functools.partial`** - creates a new function from an existing one with some arguments pre-filled. Useful when you need a simpler interface to a function that has many parameters, without rewriting it.